In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
##### import os
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# -----------------------
# Fix seeds
# -----------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()


df = pd.read_csv("/kaggle/input/amplifyduibi/AMP_amplify.csv")
print(df.head())

labels = df["label"].values
sequences = df["Sequence"].tolist()

AA = "ACDEFGHIKLMNPQRSTVWYBXZ"
def build_tokenizer():
    tok = {aa: i+1 for i, aa in enumerate(AA)}
    tok["PAD"] = 0
    tok["MASK"] = len(tok)
    return tok

tokenizer_tiny = build_tokenizer()

def tokenize_seq_tiny(seq, max_len=128):
    ids = [tokenizer_tiny.get(a, tokenizer_tiny["X"]) for a in seq]
    ids = ids[:max_len]
    return ids + [tokenizer_tiny["PAD"]] * (max_len - len(ids))



tiny_model = TinyProteinTransformer(
    vocab_size=len(tokenizer_tiny),
    num_classes=None
)

tiny_model.load_state_dict(torch.load("/kaggle/input/amplifyduibi/best_pretrain_10M.pt"))
tiny_model.cuda()
tiny_model.eval()

for p in tiny_model.parameters():
    p.requires_grad = False


# ============================================================
# 4) Load ESM2 models
# ============================================================
import esm

esm_models = {  
    "esm2_t12_35M": "esm2_t12_35M_UR50D",
    "esm2_t30_150M": "esm2_t30_150M_UR50D",
    "esm2_t33_650M": "esm2_t33_650M_UR50D",
    "esm2_t36_3B": "esm2_t36_3B_UR50D",
    
}

esm_encoders = {}
for name, tag in esm_models.items():
    model, alphabet = esm.pretrained.load_model_and_alphabet(tag)
    model = model.cuda().eval()
    for p in model.parameters(): p.requires_grad = False
    esm_encoders[name] = (model, alphabet)


# ============================================================
# 5) ProtT5 (very large, optional)
# ============================================================
from transformers import T5EncoderModel, T5Tokenizer

enable_prott5 = True  # set True if GPU allows

if enable_prott5:
    prott5_tokenizer = T5Tokenizer.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc")
    prott5_model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc")
    prott5_model.cuda().eval()
    for p in prott5_model.parameters(): p.requires_grad = False


from transformers import BertTokenizer, BertModel

enable_protbert = True

if enable_protbert:
    protbert_tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
    protbert_model = BertModel.from_pretrained("Rostlab/prot_bert")
    protbert_model.cuda().eval()
    for p in protbert_model.parameters():
        p.requires_grad = False

# ============================================================
# 6) Encode function for each backbone
# ============================================================
@torch.no_grad()
def encode_tiny(seq):
    x = torch.tensor(tokenize_seq_tiny(seq)).unsqueeze(0).cuda()
    h = tiny_model.encode(x)               # B, L, D
    pooled = tiny_model.attention_pool(h)  # B, D
    return pooled.squeeze(0)


@torch.no_grad()
def encode_esm(model, alphabet, seq):
    batch = [(0, seq)]
    data = alphabet.get_batch_converter()(batch)
    _, _, toks = data
    toks = toks.cuda()
    out = model(toks, repr_layers=[model.num_layers])["representations"][model.num_layers]
    return out.mean(1).squeeze(0)  # mean pooling


@torch.no_grad()
def encode_prott5(seq):
    seq_spaced = " ".join(list(seq)) 
    ids = prott5_tokenizer(seq_spaced, return_tensors="pt", padding=True).input_ids.cuda()
    e = prott5_model(ids).last_hidden_state
    return e.mean(1).squeeze(0)

@torch.no_grad()
def encode_protbert(seq):
    seq_spaced = " ".join(list(seq))     
    toks = protbert_tokenizer(seq_spaced, return_tensors="pt", padding=True)
    for k in toks:
        toks[k] = toks[k].cuda()

    out = protbert_model(**toks).last_hidden_state
    return out.mean(1).squeeze(0)        

# ============================================================
# 7) Training loop (linear classifier)
# ============================================================

from torch.utils.data import DataLoader, TensorDataset

def train_classifier(X_train, y_train, X_val, y_val, epochs=5, lr=1e-3, batch_size=64):
    clf = nn.Linear(X_train.size(1), 2).cuda()
    opt = torch.optim.AdamW(clf.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    # ---- build dataloader ----
    train_ds = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    best_acc = -1
    best_f1 = -1
    best_auc = -1

    for ep in range(epochs):
        clf.train()

        for xb, yb in train_loader:
            xb = xb.cuda()
            yb = yb.cuda()

            opt.zero_grad()
            out = clf(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()

        # ---- eval for this epoch ----
        clf.eval()
        with torch.no_grad():
            logits = clf(X_val.cuda())
            preds = logits.argmax(dim=1).cpu().numpy()
            probs = logits.softmax(1)[:,1].cpu().numpy()

        acc = accuracy_score(y_val.cpu().numpy(), preds)
        f1 = f1_score(y_val.cpu().numpy(), preds)
        auc = roc_auc_score(y_val.cpu().numpy(), probs)
        print(f"Epoch {ep}: acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")

        # ---- keep best ----
        if acc > best_acc:
            best_acc = acc
            best_f1 = f1
            best_auc = auc

    return best_acc, best_f1, best_auc

# ============================================================
# 8) 5-Fold CV wrapper
# ============================================================

def cross_validate_model(encode_fn, model_name):
    print(f"\n===== Running 5-Fold CV for {model_name} =====")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    accs, f1s, aucs = [], [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(sequences, labels)):
        print(f"Fold {fold+1}")

        seq_train = [sequences[i] for i in train_idx]
        seq_val   = [sequences[i] for i in val_idx]
        y_train = torch.tensor(labels[train_idx]).cuda()
        y_val   = torch.tensor(labels[val_idx]).cuda()

        # Encode all sequences (frozen encoder)
        X_train = torch.stack([encode_fn(s) for s in tqdm(seq_train)])
        X_val   = torch.stack([encode_fn(s) for s in tqdm(seq_val)])

        acc, f1, auc = train_classifier(X_train, y_train, X_val, y_val)
        accs.append(acc); f1s.append(f1); aucs.append(auc)

    acc_mean, acc_std = np.mean(accs), np.std(accs)
    f1_mean,  f1_std  = np.mean(f1s), np.std(f1s)
    auc_mean, auc_std = np.mean(aucs), np.std(aucs)

    print(f"\n{model_name} Results:")
    print(f"ACC: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"F1 : {f1_mean:.4f} ± {f1_std:.4f}")
    print(f"AUC: {auc_mean:.4f} ± {auc_std:.4f}")


    return {
        "model": model_name,
        "acc": np.mean(accs),
        "f1": np.mean(f1s),
        "auc": np.mean(aucs)
    }


# ============================================================
# 9) Run all models
# ============================================================
results = []

# TinyProteinTransformer
results.append(cross_validate_model(encode_tiny, "TinyProteinTransformer"))

# ESM models
for name, (m, alphabet) in esm_encoders.items():
    encode_fn = lambda seq, m=m, a=alphabet: encode_esm(m, a, seq)
    results.append(cross_validate_model(encode_fn, name))

# ProtT5
if enable_prott5:
    results.append(cross_validate_model(encode_prott5, "ProtT5-XL"))

if enable_protbert:
    results.append(cross_validate_model(encode_protbert, "ProtBERT"))


# ============================================================
# 10) Show result table
# ============================================================
print("\n==== Final Results Summary ====")
df_res = pd.DataFrame(results)
print(df_res)

In [ ]:
# ======================================================
# =============  0. Imports and Setup  =================
# ======================================================
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from umap.umap_ import UMAP

import esm
from transformers import T5EncoderModel, T5Tokenizer, BertTokenizer, BertModel

plt.style.use("seaborn-v0_8")
torch.set_grad_enabled(False)

# ======================================================
# =============  1. Fix Seeds  =========================
# ======================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()


# ======================================================
# =============  2. Load AMP dataset  ==================
# ======================================================
df = pd.read_csv("/kaggle/input/amplifyduibi/toxmsrc_merge.csv")
sequences = df["Sequence"].tolist()
labels = df["label"].values

print("Loaded AMP dataset:", df.shape)


# ======================================================
# =============  3. Tokenizer & Tiny Model  ============
# ======================================================
AA = "ACDEFGHIKLMNPQRSTVWYBXZ"

def build_tokenizer():
    tok = {aa: i+1 for i, aa in enumerate(AA)}
    tok["PAD"] = 0
    tok["MASK"] = len(tok)
    return tok

tokenizer_tiny = build_tokenizer()

def tokenize_seq_tiny(seq, max_len=128):
    ids = [tokenizer_tiny.get(a, tokenizer_tiny["X"]) for a in seq]
    ids = ids[:max_len]
    return ids + [tokenizer_tiny["PAD"]] * (max_len - len(ids))


# ---------------- TinyProteinTransformer ----------------
class TinyProteinTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes=None,
        hidden_dim=640,
        num_layers=20,
        num_heads=10,
        max_len=128,
        cnn_kernel_sizes=(3,5,7,9),
        cnn_out=256,
        dropout=0.05
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos   = nn.Embedding(max_len, hidden_dim)

        self.convs = nn.ModuleList([
            nn.Conv1d(hidden_dim, cnn_out, k, padding=k//2)
            for k in cnn_kernel_sizes
        ])
        self.conv_proj = nn.Linear(cnn_out * len(cnn_kernel_sizes), hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.attn_pool = nn.Linear(hidden_dim, 1)
        self.dropout   = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(hidden_dim)

    def encode(self, x):
        B, L = x.size()
        pos_ids = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)

        h = self.embed(x) + self.pos(pos_ids)
        h = self.dropout(h)

        cnn_in = h.permute(0,2,1)
        convs = [F.relu(conv(cnn_in)) for conv in self.convs]
        conv_cat = torch.cat(convs, dim=1).permute(0,2,1)
        conv_out = self.conv_proj(conv_cat)

        h = self.ln(h + conv_out)
        return self.encoder(h)

    def attention_pool(self, h):
        att = torch.softmax(self.attn_pool(h), dim=1)
        return (att * h).sum(dim=1)


# -------- Load pretrained Tiny --------
tiny_model = TinyProteinTransformer(vocab_size=len(tokenizer_tiny))
tiny_model.load_state_dict(torch.load("/kaggle/input/amplifyduibi/best_pretrain_10M.pt"),strict=False)
tiny_model.cuda().eval()
for p in tiny_model.parameters():
    p.requires_grad=False

VALID = set("ACDEFGHIKLMNPQRSTVWYBXZ")

def clean_seq(seq):
    return "".join([a if a in VALID else "X" for a in seq])
    
# ======================================================
# =============  4. Load ESM Models  ===================
# ======================================================
esm_models = {
    "esm2_t30_150M": "esm2_t30_150M_UR50D",
    "esm2_t33_650M": "esm2_t33_650M_UR50D",
}

esm_encoders = {}
for name, tag in esm_models.items():
    model, alphabet = esm.pretrained.load_model_and_alphabet(tag)
    model = model.cuda().eval()
    for p in model.parameters(): p.requires_grad=False
    esm_encoders[name] = (model, alphabet)


# ======================================================
# =============  5. Encode functions  ==================
# ======================================================
@torch.no_grad()
def encode_tiny(seq):
    x = torch.tensor(tokenize_seq_tiny(seq)).unsqueeze(0).cuda()
    h = tiny_model.encode(x)
    return tiny_model.attention_pool(h).squeeze(0)


@torch.no_grad()
def encode_esm(model, alphabet, seq):
    batch = [(0, seq)]
    data = alphabet.get_batch_converter()(batch)
    _, _, toks = data
    toks = toks.cuda()
    out = model(toks, repr_layers=[model.num_layers])["representations"][model.num_layers]
    return out.mean(1).squeeze(0)


# ======================================================
# =============  6. Build visualization subset =========
# ======================================================
N = 7513
idx = np.random.choice(len(sequences), N, replace=False)
seq_vis  = [sequences[i] for i in idx]
lab_vis  = np.array([labels[i] for i in idx])


# ======================================================
# =============  7. Batch Encoding  ====================
# ======================================================
def batch_encode(seqs, fn):
    embs = []
    for s in tqdm(seqs):
        embs.append(fn(s).cpu().numpy())
    return np.array(embs)

print("Encoding Tiny...")
emb_tiny = batch_encode(seq_vis, encode_tiny)

print("Encoding ESM150...")
emb_150 = batch_encode(seq_vis, lambda s: encode_esm(*esm_encoders["esm2_t30_150M"], clean_seq(s)))

print("Encoding ESM650...")
emb_650 = batch_encode(seq_vis, lambda s: encode_esm(*esm_encoders["esm2_t33_650M"], clean_seq(s)))


# ======================================================
# =============  8. Dimensionality Reduction  ==========
# ======================================================
emb_tiny32 = emb_tiny.astype(np.float32)
emb_150_32 = emb_150.astype(np.float32)
emb_650_32 = emb_650.astype(np.float32)

tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
t_tiny = tsne.fit_transform(emb_tiny32)
t_150  = tsne.fit_transform(emb_150_32)
t_650  = tsne.fit_transform(emb_650_32)


# ======================================================
# =============  9. Plotting  ==========================
# ======================================================
def plot_embed1(embed, labels, title):
    plt.figure(figsize=(7,5))
    plt.scatter(embed[:,0], embed[:,1],
                c=labels, cmap="coolwarm",
                s=14, alpha=0.8, edgecolor="none")
    plt.title(title, fontsize=16)
    plt.xticks([]); plt.yticks([])
    plt.tight_layout()
    plt.savefig("/kaggle/working/tsne_tox_tpt.pdf",
                format="pdf",
                bbox_inches="tight")
    plt.show()
    plt.close()

def plot_embed2(embed, labels, title):
    plt.figure(figsize=(7,5))
    plt.scatter(embed[:,0], embed[:,1],
                c=labels, cmap="coolwarm",
                s=14, alpha=0.8, edgecolor="none")
    plt.title(title, fontsize=16)
    plt.xticks([]); plt.yticks([])
    plt.tight_layout()
    plt.savefig("/kaggle/working/tsne_tox_esm150.pdf",
                format="pdf",
                bbox_inches="tight")
    plt.show()
    plt.close()
    
# ---------------- t-SNE ----------------
plot_embed1(t_tiny, lab_vis, "t-SNE – TinyProteinTransformer (tox)")
plot_embed2(t_150,  lab_vis, "t-SNE – ESM2-150M (tox)")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# AMP AUC data
models = ["TPT (103M)", "ESM2-t12 (35M)", "ESM2-t30 (150M)", "ProtBERT (420M)"]
auc = [0.9282, 0.9183, 0.9273, 0.9076]
param_m = [103, 35, 150, 420]

df = pd.DataFrame({
    "Model": models,
    "Param_M": param_m,
    "AUC": auc
})

plt.figure(figsize=(10, 7))

colors = plt.cm.tab10(range(len(df)))

for i, row in df.iterrows():
    plt.scatter(
        row["Param_M"],
        row["AUC"],
        s=200,
        color=colors[i],
        label=row["Model"]
    )

plt.xlabel("Parameter Size (Millions)", fontsize=14)
plt.ylabel("AUC (AMP)", fontsize=14)
plt.title("AMP Task: AUC vs Model Parameter Size", fontsize=16)

plt.legend(
    loc="upper right",
    fontsize=12,
    frameon=True
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/auc_size.pdf",
                format="pdf",
                bbox_inches="tight")
plt.show()
